# read checkpoint from gabor

In [1]:
import torch
from gaussianimage_cholesky import GaussianImage_Cholesky

In [2]:
def read_checkpoint(model_name, model_path, num_points, H, W, device):
    if model_name == "GaussianImage_Cholesky":
        from gaussianimage_cholesky import GaussianImage_Cholesky
        gaussian_model = GaussianImage_Cholesky(loss_type="L2", opt_type="adan", num_points=num_points, H=H, W=W, BLOCK_H=16, BLOCK_W=16, 
                device=device, lr=0.05, quantize=False).to(device)

    elif model_name == "GaussianImage_RS":
        from gaussianimage_rs import GaussianImage_RS
        gaussian_model = GaussianImage_RS(loss_type="L2", opt_type="adan", num_points=num_points, H=H, W=W, BLOCK_H=16, BLOCK_W=16, 
                device=device, lr=0.05, quantize=False).to(device) 

    elif model_name == "3DGS":
        from gaussiansplatting_3d import Gaussian3D
        gaussian_model = Gaussian3D(loss_type="Fusion2", opt_type="adan", num_points=num_points, H=H, W=W, BLOCK_H=16, BLOCK_W=16, 
                device=device, sh_degree=sh_degree, lr=0.05).to(device)
        
    if model_path is not None:
        print(f"loading model path:{model_path}")
        checkpoint = torch.load(model_path, map_location=device)
        model_dict = gaussian_model.state_dict()
        pretrained_dict = {k: v for k, v in checkpoint.items() if k in model_dict}
        model_dict.update(pretrained_dict)
        gaussian_model.load_state_dict(model_dict)
        return gaussian_model

In [3]:
H, W = 512, 768
device = torch.device("cpu")
gabor = read_checkpoint("GaussianImage_Cholesky", "checkpoints/kodak/GaussianImage_Cholesky_50000_1000/kodim01/gaussian_model.pth.tar", 1000,H, W, device)

loading model path:checkpoints/kodak/GaussianImage_Cholesky_50000_1000/kodim01/gaussian_model.pth.tar


In [4]:
state_dict = gabor.state_dict()
for name, param in state_dict.items():
    print(name, param.shape)

_xyz torch.Size([1000, 2])
_cholesky torch.Size([1000, 3])
_features_dc torch.Size([1000, 3])
gabor_freqs torch.Size([4000, 2])
gabor_weights torch.Size([4000, 1])
_opacity torch.Size([1000, 1])
background torch.Size([3])
bound torch.Size([1, 2])
cholesky_bound torch.Size([1, 3])


In [5]:
print(state_dict["_xyz"])

tensor([[ 0.6459, -0.5576],
        [-0.4180,  0.8397],
        [-1.6880,  0.6282],
        ...,
        [-1.2354,  1.7766],
        [ 0.7299,  1.3873],
        [-0.4381, -0.6427]])


In [6]:
import pandas as pd

params = [(name, tuple(param.shape)) for name, param in gabor.state_dict().items()]
df = pd.DataFrame(params, columns=["Parameter", "Shape"])
df

,Parameter,Shape
0,_xyz,"(1000, 2)"
1,_cholesky,"(1000, 3)"
2,_features_dc,"(1000, 3)"
3,gabor_freqs,"(4000, 2)"
4,gabor_weights,"(4000, 1)"
5,_opacity,"(1000, 1)"
6,background,"(3,)"
7,bound,"(1, 2)"
8,cholesky_bound,"(1, 3)"


In [7]:
params = state_dict["gabor_weights"]
weights = pd.DataFrame(params, columns=["gabor weights"])
weights.describe()

,gabor weights
count,4000.000000
mean,-1.791121
std,3.859759
min,-27.049999
25%,-1.642333
50%,-0.648471
75%,-0.366915
max,12.618017


In [8]:
freqs = pd.DataFrame(state_dict["gabor_freqs"], columns=["x", "y"] )
freqs

,x,y
0,0.130200,-0.056203
1,0.210023,0.228303
2,-0.048931,-0.041127
3,0.046380,-0.173430
4,-0.018204,-0.012457
...,...,...
3995,-0.078691,0.179707
3996,-0.031077,0.035507
3997,-0.031077,0.035507
3998,-0.031077,0.035507


In [9]:
freqs.describe()

,x,y
count,4000.000000,4000.000000
mean,0.028343,0.025545
std,0.228985,0.253532
min,-1.830510,-2.264869
25%,-0.033695,-0.053091
50%,0.038501,0.035827
75%,0.098722,0.098814
max,1.530262,2.503212


In [10]:
colors = pd.DataFrame(state_dict["_features_dc"], columns=["r", "g", "b"] )
colors

,r,g,b
0,0.686144,0.701858,0.565457
1,0.542213,0.315687,0.234240
2,0.734256,0.763123,0.610821
3,0.962359,0.555671,0.342695
4,0.585867,0.457381,0.365265
...,...,...,...
995,0.614620,0.261957,0.138769
996,0.420590,0.411683,0.321622
997,0.398297,0.399402,0.350951
998,-0.175586,-0.202325,-0.184041


In [11]:
colors.describe()

,r,g,b
count,1000.000000,1000.000000,1000.000000
mean,0.609375,0.543354,0.433086
std,0.699343,0.684675,0.606036
min,-4.410773,-4.399441,-4.027427
25%,0.466323,0.383573,0.297458
50%,0.617530,0.554514,0.442610
75%,0.723001,0.678956,0.543443
max,12.571785,13.251416,12.192554


In [12]:
opac = pd.DataFrame(state_dict["_opacity"], columns = ["opacity"])
opac.describe()

,opacity
count,1000.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0
